In [1]:
#TEST SPECIFIC SPATIAL EXTENT - GLOBAL SETTINGS FOR THE SPATIAL EXTENT TO TEST COMPUTE()

# Specify the spatial extent (bounding box)
spatial_extent = {
    "west": 8,
    "east": 14,
    "south": 42,
    "north": 50
}

region = {"lon": slice(8, 14), "lat": slice(50, 42)}


In [2]:
import xarray as xr
import dask.array as da
from lightgbm import LGBMRegressor
from dask.distributed import LocalCluster, Client, performance_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import psutil
import os
from dask.diagnostics import ProgressBar

import warnings

warnings.filterwarnings(
    "ignore",
    module="sklearn.utils.validation"
)


cluster = LocalCluster(
    n_workers=1,  # Fewer workers = fewer WebSocket connections
    threads_per_worker=16,
    worker_dashboard_address=False,
    diagnostics_port=None# Disable per-worker dashboards
)
client = Client(cluster)    
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 1
Total threads: 16,Total memory: 98.23 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36499,Workers: 1
Dashboard: http://127.0.0.1:8787/status,Total threads: 16
Started: Just now,Total memory: 98.23 GiB
Comm: tcp://127.0.0.1:36883,Total threads: 16
Dashboard: http://127.0.0.1:33137/status,Memory: 98.23 GiB
Nanny: tcp://127.0.0.1:39507,


In [3]:
from openeo.local import LocalConnection

# Initialize the local connection
local_conn = LocalConnection("./")

# Define the STAC collection URL
stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"

# Specify the temporal extent
temporal_extent = ["2000-01-01", "2020-12-31"]

# Load the data cube with specified parameters
era5_single = local_conn.load_stac(
    url=stac_item,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["data"]
).execute()

# Convert to xarray Dataset and select the 't2m' variable
era5_single = era5_single.to_dataset(dim='bands')["t2m"].to_dataset()

# Display the dataset
era5_single


<xarray.Dataset> Size: 25MB
Dimensions:  (lat: 33, lon: 25, time: 7670)
Coordinates:
  * lat      (lat) float64 264B 50.0 49.75 49.5 49.25 ... 42.75 42.5 42.25 42.0
  * lon      (lon) float64 200B 8.0 8.25 8.5 8.75 9.0 ... 13.25 13.5 13.75 14.0
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    t2m      (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>

In [4]:
stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

era5_pressure = local_conn.load_stac(
    url=stac_item,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["data"]
).execute()
era5_pressure = era5_pressure.to_dataset(dim='bands')
era5_pressure

<xarray.Dataset> Size: 127MB
Dimensions:  (time: 7670, lat: 33, lon: 25)
Coordinates:
  * lat      (lat) float64 264B 50.0 49.75 49.5 49.25 ... 42.75 42.5 42.25 42.0
  * lon      (lon) float64 200B 8.0 8.25 8.5 8.75 9.0 ... 13.25 13.5 13.75 14.0
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    q_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
    t_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
    u_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
    v_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
    z_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    crs:                     EPSG:4326
    history:                 2024-11-15T16:05 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [5]:
ERA5 = xr.merge([era5_single, era5_pressure])
ERA5

<xarray.Dataset> Size: 152MB
Dimensions:  (lat: 33, lon: 25, time: 7670)
Coordinates:
  * lat      (lat) float64 264B 50.0 49.75 49.5 49.25 ... 42.75 42.5 42.25 42.0
  * lon      (lon) float64 200B 8.0 8.25 8.5 8.75 9.0 ... 13.25 13.5 13.75 14.0
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    t2m      (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
    q_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
    t_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
    u_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
    v_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>
    z_850    (time, lat, lon) float32 25MB dask.array<chunksize=(500, 33, 25), meta=np.ndarray>

In [6]:
stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

emo1 = local_conn.load_stac(
    url=stac_item,
    bands=["data"],
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
).execute()
EMO1 = emo1.to_dataset(dim='bands')["ta24"].to_dataset()
EMO1

<xarray.Dataset> Size: 11GB
Dimensions:  (lat: 480, lon: 360, time: 7670)
Coordinates:
  * lat      (lat) float64 4kB 49.99 49.97 49.96 49.94 ... 42.04 42.02 42.01
  * lon      (lon) float64 3kB 8.008 8.025 8.042 8.058 ... 13.96 13.98 13.99
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    ta24     (time, lat, lon) float64 11GB dask.array<chunksize=(365, 8, 24), meta=np.ndarray>

In [7]:
stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

dem = local_conn.load_stac(
    url=stac_item,
    spatial_extent=spatial_extent,
    bands=["data"]
).execute()
dem = dem.to_dataset(dim='bands')["dem"].to_dataset()
dem

<xarray.Dataset> Size: 1MB
Dimensions:  (lat: 480, lon: 360, time: 1)
Coordinates:
  * lat      (lat) float64 4kB 49.99 49.97 49.96 49.94 ... 42.04 42.02 42.01
  * lon      (lon) float64 3kB 8.008 8.025 8.042 8.058 ... 13.96 13.98 13.99
  * time     (time) datetime64[ns] 8B 2000-01-01
Data variables:
    dem      (time, lat, lon) float64 1MB dask.array<chunksize=(1, 61, 181), meta=np.ndarray>

In [8]:
dem.lat

<xarray.DataArray 'lat' (lat: 480)> Size: 4kB
array([49.991667, 49.975   , 49.958333, ..., 42.041667, 42.025   , 42.008333])
Coordinates:
  * lat      (lat) float64 4kB 49.99 49.97 49.96 49.94 ... 42.04 42.02 42.01
Attributes:
    axis:           Y
    long_name:      latitude
    standard_name:  latitude
    units:          degrees_north

In [9]:
import xarray as xr



single = xr.open_zarr("/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/EMO1_DOWNSCALING/data/SEAS5_AUGUST_2021_SINGLE.zarr/", chunks={}).sel(region)
pressure = xr.open_zarr("/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/EMO1_DOWNSCALING/data/SEAS5_AUGUST_2021_PRESSURE.zarr/", chunks={}).sel(region)
SEAS5 = xr.merge([single["t2m"], pressure])
SEAS5

<xarray.Dataset> Size: 218MB
Dimensions:  (lat: 33, lon: 25, number: 51, time: 216)
Coordinates:
  * lat      (lat) float32 132B 50.0 49.75 49.5 49.25 ... 42.75 42.5 42.25 42.0
  * lon      (lon) float32 100B 8.0 8.25 8.5 8.75 9.0 ... 13.25 13.5 13.75 14.0
  * number   (number) int32 204B 0 1 2 3 4 5 6 7 8 ... 43 44 45 46 47 48 49 50
  * time     (time) datetime64[ns] 2kB 2021-08-01 2021-08-02 ... 2022-03-04
Data variables:
    t2m      (time, number, lat, lon) float32 36MB dask.array<chunksize=(216, 1, 33, 25), meta=np.ndarray>
    q_850    (time, number, lat, lon) float32 36MB dask.array<chunksize=(216, 1, 33, 25), meta=np.ndarray>
    t_850    (time, number, lat, lon) float32 36MB dask.array<chunksize=(216, 1, 33, 25), meta=np.ndarray>
    u_850    (time, number, lat, lon) float32 36MB dask.array<chunksize=(216, 1, 33, 25), meta=np.ndarray>
    v_850    (time, number, lat, lon) float32 36MB dask.array<chunksize=(216, 1, 33, 25), meta=np.ndarray>
    z_850    (time, number, lat, lon) float32 36MB dask.array<chunksize=(216, 1, 33, 25), meta=np.ndarray>
Attributes:
    long_name:  2 metre temperature
    units:      K

## REMAPPING

In [10]:
import xarray as xr
import numpy as np

def match_to_mid_resolution(source_ds, target_ds, lat_name='lat', lon_name='lon', num_mid_lats=int(len(dem["lat"]) // 2), num_mid_lons=int(len(dem["lon"]) // 2)):
    # Get coordinate bounds from union of source and target
    min_lat = max(source_ds[lat_name].min().item(), target_ds[lat_name].min().item())
    max_lat = min(source_ds[lat_name].max().item(), target_ds[lat_name].max().item())
    min_lon = max(source_ds[lon_name].min().item(), target_ds[lon_name].min().item())
    max_lon = min(source_ds[lon_name].max().item(), target_ds[lon_name].max().item())
    
    # Create mid-resolution grid
    mid_lats = np.linspace(min_lat, max_lat, num_mid_lats)
    mid_lons = np.linspace(min_lon, max_lon, num_mid_lons)
    
    # Interpolate both datasets to mid-resolution grid using bilinear interpolation
    mid_coords = {
        lat_name: xr.DataArray(mid_lats, dims=lat_name),
        lon_name: xr.DataArray(mid_lons, dims=lon_name)
    }
    
    source_mid = source_ds.interp(mid_coords, method='linear')
    
    return source_mid

SEAS5_mid = match_to_mid_resolution(SEAS5, dem).astype('float32')
ERA5_mid = match_to_mid_resolution(ERA5, dem).astype('float32')
EMO1_mid =  match_to_mid_resolution(EMO1, dem).astype('float32')


Normalised Min-Max

In [11]:
import xarray as xr

def normalize_dataset(ds):
    # Create a copy to avoid modifying the original dataset
    ds_normalized = ds.copy()
    
    # Loop through all data variables
    for var in ds.data_vars:
        # Subtract the minimum (along all dimensions except the variable's own)
        min_val = ds[var].min(keep_attrs=True)
        ds_normalized[var] = ds[var] - min_val
        
        # Divide by the maximum (after min subtraction)
        max_val = ds_normalized[var].max(keep_attrs=True)
        ds_normalized[var] = ds_normalized[var] / max_val
        
        # Preserve attributes if they exist
        if 'attrs' in ds[var].attrs:
            ds_normalized[var].attrs.update(ds[var].attrs)
    
    return ds_normalized

# Usage:
SEAS5_mid_normalized = normalize_dataset(SEAS5_mid)
ERA5_mid_normalized = normalize_dataset(ERA5_mid)

In [12]:
SEAS5_mid_normalized

<xarray.Dataset> Size: 11GB
Dimensions:  (time: 216, number: 51, lat: 240, lon: 180)
Coordinates:
  * number   (number) int32 204B 0 1 2 3 4 5 6 7 8 ... 43 44 45 46 47 48 49 50
  * time     (time) datetime64[ns] 2kB 2021-08-01 2021-08-02 ... 2022-03-04
  * lat      (lat) float64 2kB 42.01 42.04 42.08 42.11 ... 49.92 49.96 49.99
  * lon      (lon) float64 1kB 8.008 8.042 8.075 8.109 ... 13.92 13.96 13.99
Data variables:
    t2m      (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    q_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    t_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    u_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    v_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    z_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
Attributes:
    long_name:  2 metre temperature
    units:      K

Cyclic Feature Addition

In [13]:
import numpy as np
import dask.array as da
from datetime import date


def encode_cyclical_features(values, max_value):
    """Encode cyclical features using sine and cosine transformations."""
    sin = np.sin(2 * np.pi * values / max_value)
    cos = np.cos(2 * np.pi * values / max_value)
    return sin, cos

def repeat_along_axis(arr, repeats, axis):
    """Repeat array along specified axis."""
    return da.repeat(arr[None, ...], repeats, axis=axis)

def get_spatial_dims(ds):
    """
    Detect spatial dimension names in the dataset.
    Returns (y_dim, x_dim) tuple based on common naming conventions.
    """
    dims = set(ds.dims)
    
    y_candidates = ['y', 'lat', 'latitude', 'lats']
    x_candidates = ['x', 'lon', 'longitude', 'long', 'lons']
    
    y_dim = next((d for d in y_candidates if d in dims), None)
    x_dim = next((d for d in x_candidates if d in dims), None)
    
    if y_dim is None or x_dim is None:
        raise ValueError(
            f"Could not detect spatial dimensions. Available dimensions: {list(dims)}. "
            f"Tried y names: {y_candidates}, x names: {x_candidates}"
        )
    
    return y_dim, x_dim

def get_existing_chunks(ds, dims):
    """
    Get chunking pattern from existing variables in the dataset.
    Returns dict of {dim: chunksize} for the specified dimensions.
    """
    chunks = {}
    for var in ds.data_vars.values():
        if hasattr(var.data, 'chunks'):
            var_chunks = dict(zip(var.dims, var.data.chunks))
            for dim in dims:
                if dim in var_chunks and dim not in chunks:
                    # Take first chunk size found for each dimension
                    chunks[dim] = var_chunks[dim][0]
        if all(dim in chunks for dim in dims):
            break
    return chunks or None

def encode_doys(ds, time_dim='time', spatial_dims=None, extra_dims=None, inplace=False):
    """
    Encode day of year as cyclical features and add to dataset, handling:
    - ERA5 (time, y, x)
    - SEAS5 (time, number, y, x) or similar
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset containing time dimension
    time_dim : str, optional
        Name of time dimension (default: 'time')
    spatial_dims : tuple, optional
        Explicit (y_dim, x_dim) names. If None, auto-detect.
    extra_dims : list, optional
        Additional dimensions (e.g., ['number']) to repeat along
    inplace : bool, optional
        If True, modify dataset in place (default: False)
    
    Returns:
    --------
    xarray.Dataset
        Dataset with sin_doy and cos_doy variables added
    """
    if not inplace:
        ds = ds.copy()
    
    # Auto-detect spatial dims if not provided
    if spatial_dims is None:
        y_dim, x_dim = get_spatial_dims(ds)
    else:
        y_dim, x_dim = spatial_dims
    
    # Determine all target dimensions
    target_dims = [time_dim]
    if extra_dims:
        target_dims.extend(extra_dims)
    target_dims.extend([y_dim, x_dim])
    
    # Get chunking pattern from existing variables
    chunks = get_existing_chunks(ds, target_dims)
    
    # Compute day of year
    doys = ds[time_dim].values.astype('datetime64[D]')
    doys = da.asarray([date.timetuple(doy.astype(object)).tm_yday for doy in doys])
    
    # Encode cyclical features
    sin_doy, cos_doy = encode_cyclical_features(doys, 365)
    
    # Reshape and repeat along all non-time dimensions
    for dim in target_dims[1:]:  # Skip time_dim
        repeats = len(ds[dim])
        sin_doy = da.repeat(sin_doy[..., None], repeats, axis=-1)
        cos_doy = da.repeat(cos_doy[..., None], repeats, axis=-1)
    
    # Reshape to final dimensions
    sin_doy = sin_doy.reshape([len(ds[dim]) for dim in target_dims])
    cos_doy = cos_doy.reshape([len(ds[dim]) for dim in target_dims])
    
    # Apply chunking if found
    if chunks:
        chunk_sizes = [chunks.get(dim, -1) for dim in target_dims]
        sin_doy = sin_doy.rechunk(chunk_sizes)
        cos_doy = cos_doy.rechunk(chunk_sizes)
    
    # Add to dataset
    ds['sin_doy'] = (target_dims, sin_doy)
    ds['cos_doy'] = (target_dims, cos_doy)
    
    # Add attributes
    for name in ['sin_doy', 'cos_doy']:
        ds[name].attrs.update({
            'long_name': f"{'Sine' if 'sin' in name else 'Cosine'} of day of year",
            'units': 'unitless',
            'description': f"Cyclical encoding of day of year"
        })
    
    return ds

    # For ERA5 (only time, y, x)
encode_doys(ERA5_mid_normalized, inplace=True)

# For SEAS5 (time, number, y, x)
encode_doys(
    SEAS5_mid_normalized,
    extra_dims=['number'],  # or 'ensemble' depending on SEAS5's dim name
    inplace=True
)


<xarray.Dataset> Size: 19GB
Dimensions:  (time: 216, number: 51, lat: 240, lon: 180)
Coordinates:
  * number   (number) int32 204B 0 1 2 3 4 5 6 7 8 ... 43 44 45 46 47 48 49 50
  * time     (time) datetime64[ns] 2kB 2021-08-01 2021-08-02 ... 2022-03-04
  * lat      (lat) float64 2kB 42.01 42.04 42.08 42.11 ... 49.92 49.96 49.99
  * lon      (lon) float64 1kB 8.008 8.042 8.075 8.109 ... 13.92 13.96 13.99
Data variables:
    t2m      (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    q_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    t_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    u_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    v_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    z_850    (time, number, lat, lon) float32 2GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    sin_doy  (time, number, lat, lon) float64 4GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
    cos_doy  (time, number, lat, lon) float64 4GB dask.array<chunksize=(216, 1, 240, 180), meta=np.ndarray>
Attributes:
    long_name:  2 metre temperature
    units:      K

In [14]:
ERA5_mid_normalized

<xarray.Dataset> Size: 13GB
Dimensions:  (time: 7670, lat: 240, lon: 180)
Coordinates:
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
  * lat      (lat) float64 2kB 42.01 42.04 42.08 42.11 ... 49.92 49.96 49.99
  * lon      (lon) float64 1kB 8.008 8.042 8.075 8.109 ... 13.92 13.96 13.99
Data variables:
    t2m      (time, lat, lon) float32 1GB dask.array<chunksize=(500, 240, 180), meta=np.ndarray>
    q_850    (time, lat, lon) float32 1GB dask.array<chunksize=(500, 240, 180), meta=np.ndarray>
    t_850    (time, lat, lon) float32 1GB dask.array<chunksize=(500, 240, 180), meta=np.ndarray>
    u_850    (time, lat, lon) float32 1GB dask.array<chunksize=(500, 240, 180), meta=np.ndarray>
    v_850    (time, lat, lon) float32 1GB dask.array<chunksize=(500, 240, 180), meta=np.ndarray>
    z_850    (time, lat, lon) float32 1GB dask.array<chunksize=(500, 240, 180), meta=np.ndarray>
    sin_doy  (time, lat, lon) float64 3GB dask.array<chunksize=(500, 240, 180), meta=np.ndarray>
    cos_doy  (time, lat, lon) float64 3GB dask.array<chunksize=(500, 240, 180), meta=np.ndarray>

In [16]:
# Align all datasets to ensure consistent coordinates
train_X, train_y = xr.align(ERA5_mid_normalized, EMO1_mid)
#test_X = SEAS5_mid_normalized.reindex_like(train_X, method='nearest')

test_X = SEAS5_mid_normalized